# Corporate Credit Portfolio Monitor

Quarterly monitoring pack for a synthetic corporate loan book (500 obligors, 10 sectors, 2021Q1 to 2025Q4).

**This notebook contains no analytics.** Every number comes from a SQL view in `sql/`; Python only queries those views and draws them. Thresholds (PSI bands, watchlist rules, baseline quarter) are read from the `config` table. Method and limitations: `METHODOLOGY.md`.


In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display
from matplotlib.colors import LinearSegmentedColormap, PowerNorm

DB = Path("data/portfolio.duckdb")
con = duckdb.connect(str(DB), read_only=True)


def q(sql, params=None):
    """Run SQL against the monitor database and return a DataFrame."""
    return con.execute(sql, params or []).df()


cfg = dict(con.execute("SELECT key, COALESCE(value_txt, CAST(value_num AS VARCHAR)) FROM config").fetchall())
BASELINE, ASOF = cfg["baseline_quarter"], cfg["asof_quarter"]
PSI_AMBER, PSI_RED = float(cfg["psi_amber"]), float(cfg["psi_red"])

# Design tokens (validated reference palette): categorical slots 1-2, the blue
# sequential ramp for magnitude, and the fixed status colours for the RAG bands.
INK, INK_2, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e4e3df"
CURRENT, BASE = "#2a78d6", "#eb6834"
SEQ = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
RAG = {"GREEN": ("#0ca30c", "#0b0b0b"), "AMBER": ("#fab219", "#0b0b0b"), "RED": ("#d03b3b", "#ffffff")}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
})
pd.set_option("display.max_columns", 40)
display(Markdown(f"Reporting quarter **{ASOF}**, baseline **{BASELINE}**, database `{DB}`."))

## 1. Portfolio KPIs

In [ ]:
kpi = q("""
    SELECT e.quarter, e.n_obligors, e.ead_usd_mn, e.utilisation, e.expected_loss_bps,
           h.hhi, h.concentration_band, t.cumulative_share_ead AS top_n_share
    FROM v_exposure_by_quarter e
    JOIN v_hhi_by_quarter h USING (quarter)
    JOIN v_top10_concentration t USING (quarter)
    WHERE t.rank = (SELECT MAX(rank) FROM v_top10_concentration)
      AND e.quarter IN (?, ?)
    ORDER BY e.quarter
""", [BASELINE, ASOF]).set_index("quarter")

labels = {
    "n_obligors": ("Performing obligors", "{:,.0f}"),
    "ead_usd_mn": ("EAD (USD mn)", "{:,.0f}"),
    "utilisation": ("Utilisation (drawn / committed)", "{:.1%}"),
    "expected_loss_bps": ("Expected loss (bps of EAD)", "{:.1f}"),
    "hhi": ("Sector HHI (0-1)", "{:.3f}"),
    "concentration_band": ("Sector concentration band", "{}"),
    "top_n_share": ("Top-10 single-name share of EAD", "{:.1%}"),
}
view = pd.DataFrame({qtr: [fmt.format(kpi.loc[qtr, col]) for col, (_, fmt) in labels.items()] for qtr in kpi.index},
                    index=[name for name, _ in labels.values()])
view.index.name = "KPI"
display(view.style.set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

## 2. Rating distribution: reporting quarter vs baseline

Share of performing obligors by rating notch. Higher notch = weaker credit.

In [ ]:
rd = q("""
    SELECT quarter, rating, notch, share_obligors
    FROM v_rating_distribution_by_quarter WHERE quarter IN (?, ?) ORDER BY notch
""", [BASELINE, ASOF])
wide = rd.pivot(index=["notch", "rating"], columns="quarter", values="share_obligors").reset_index()

x = np.arange(len(wide))
w = 0.38
fig, ax = plt.subplots(figsize=(11, 4.2))
# 2px surface gap between the paired bars keeps them separable without outlines.
ax.bar(x - w / 2, wide[BASELINE], w, color=BASE, label=f"Baseline {BASELINE}", edgecolor=SURFACE, linewidth=1.5)
ax.bar(x + w / 2, wide[ASOF], w, color=CURRENT, label=f"Current {ASOF}", edgecolor=SURFACE, linewidth=1.5)
ax.set_xticks(x, wide["rating"], rotation=0, fontsize=8.5)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_ylabel("Share of obligors")
ax.set_title("Rating distribution of performing obligors")
ax.grid(axis="x", visible=False)
ax.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

## 3. Rating transition matrix

Rows: rating at the start of the window. Columns: rating at the end. Cell: share of the row's obligors. Colour scale is square-root stretched so the thin off-diagonal migrations stay visible next to the dominant diagonal; hatched rows had no obligors.

In [ ]:
tm = q("SELECT * FROM v_transition_matrix")
win_from, win_to = tm["from_quarter"].iloc[0], tm["to_quarter"].iloc[0]
prob = tm.pivot(index=["from_notch", "from_rating"], columns=["to_notch", "to_rating"], values="prob")
n_row = tm.drop_duplicates("from_rating").set_index("from_rating")["n_obligors_in_row"]

cmap = LinearSegmentedColormap.from_list("blue_seq", SEQ)
cmap.set_bad("#e4e3df")
data = prob.to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(11, 7.2))
im = ax.imshow(np.ma.masked_invalid(data), cmap=cmap, norm=PowerNorm(0.5, vmin=0, vmax=1), aspect="auto")
ax.set_xticks(range(data.shape[1]), [c[1] for c in prob.columns], fontsize=8.5)
ax.set_yticks(range(data.shape[0]), [f"{r[1]}  (n={int(n_row[r[1]])})" for r in prob.index], fontsize=8.5)
ax.xaxis.tick_top()
ax.xaxis.set_label_position("top")
ax.set_xlabel(f"Rating at {win_to}")
ax.set_ylabel(f"Rating at {win_from}")
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
for i in range(data.shape[0]):
    if np.isnan(data[i]).all():
        ax.add_patch(plt.Rectangle((-0.5, i - 0.5), data.shape[1], 1, fill=False, hatch="///", edgecolor=SURFACE, linewidth=0))
    for j in range(data.shape[1]):
        v = data[i, j]
        if v > 0:
            ax.text(j, i, f"{v:.0%}" if v >= 0.01 else "<1%", ha="center", va="center", fontsize=7,
                    color="#ffffff" if v ** 0.5 > 0.55 else INK)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cb.ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
cb.outline.set_visible(False)
ax.set_title(f"Empirical rating migration, {win_from} to {win_to}", pad=34)
plt.tight_layout()
plt.show()

## 4. PD back-test: expected vs realised

In [ ]:
bt = q("SELECT * FROM v_pd_backtest WHERE cohort_quarter = (SELECT MAX(cohort_quarter) FROM v_pd_backtest)")
cohort = bt["cohort_quarter"].iloc[0]
display(Markdown(
    f"**Latest full cohort: {cohort}.** Obligors performing at {cohort}; realised = defaults in the following "
    f"{cfg['backtest_horizon_q'].rstrip('0').rstrip('.')} quarters. p-value is a one-sided exact binomial test "
    f"(small = PDs too low). `low power` means fewer than 1 default was expected, so a PASS carries little information."))

show = bt[["bucket", "n_obligors", "expected_pd", "realised_default_rate", "expected_defaults", "realised_defaults",
           "realised_to_expected", "p_value_one_sided", "low_power", "test_result"]].rename(columns={
    "bucket": "Bucket", "n_obligors": "Obligors", "expected_pd": "Expected PD", "realised_default_rate": "Realised rate",
    "expected_defaults": "Exp. defaults", "realised_defaults": "Real. defaults", "realised_to_expected": "Realised / expected",
    "p_value_one_sided": "p-value", "low_power": "Low power", "test_result": "Result"})
display(show.style.hide(axis="index").format({
    "Expected PD": "{:.2%}", "Realised rate": "{:.2%}", "Exp. defaults": "{:.2f}", "Realised / expected": "{:.2f}x",
    "p-value": "{:.3f}"}, na_rep="n/a"))

display(Markdown("**All cohorts where the test rejected** (cohorts overlap, so these are not independent):"))
breaches = q("""
    SELECT cohort_quarter, bucket, n_obligors, expected_defaults, realised_defaults, p_value_one_sided, low_power
    FROM v_pd_backtest WHERE test_result = 'UNDER_PREDICTED' ORDER BY cohort_quarter, bucket
""")
display(breaches.style.hide(axis="index").format({"expected_defaults": "{:.2f}", "p_value_one_sided": "{:.3f}"}))

## 5. Population Stability Index

PSI of each quarter against the baseline. Bands from `config`: green below the amber threshold, amber up to the red threshold, red above.

In [ ]:
psi = q("""
    SELECT 'Rating' AS dimension, basis, quarter, MAX(psi_total) AS psi, MAX(psi_band) AS band FROM v_psi_rating GROUP BY ALL
    UNION ALL
    SELECT 'Sector', basis, quarter, MAX(psi_total), MAX(psi_band) FROM v_psi_sector GROUP BY ALL
    ORDER BY quarter
""")
psi["col"] = psi["dimension"] + " / " + psi["basis"].replace({"OBLIGOR_COUNT": "obligors"})
psi = psi[psi["quarter"] > BASELINE]
vals = psi.pivot(index="quarter", columns="col", values="psi")
bands = psi.pivot(index="quarter", columns="col", values="band")


def paint(_):
    out = pd.DataFrame("", index=bands.index, columns=bands.columns)
    for c in bands.columns:
        for i in bands.index:
            bg, fg = RAG[bands.loc[i, c]]
            out.loc[i, c] = f"background-color:{bg};color:{fg};font-weight:600"
    return out


# Label as well as colour, so the band survives greyscale printing and colour-blind viewing.
text = vals.copy().astype(object)
for c in vals.columns:
    for i in vals.index:
        text.loc[i, c] = f"{vals.loc[i, c]:.3f}  {bands.loc[i, c]}"
display(Markdown(f"Baseline **{BASELINE}** · green &lt; {PSI_AMBER} · amber {PSI_AMBER} to {PSI_RED} · red &gt; {PSI_RED}"))
display(text.style.apply(paint, axis=None))

## 6. Watchlist

In [ ]:
wl = q("SELECT * FROM v_watchlist")
display(Markdown(
    f"**{len(wl)} obligors flagged at {ASOF}.** "
    f"R1 downgrade: {int(wl.flag_downgrade.sum())} · R2 utilisation: {int(wl.flag_utilisation.sum())} · "
    f"R3 sector: {int(wl.flag_sector.sum())}. Sorted by number of rules tripped, then EAD. Top 25 shown."))
cols = ["obligor_id", "obligor_name", "sector", "rating_then", "rating_now", "util_prev", "util_now", "sector_ratio",
        "n_flags", "reasons", "ead_usd_mn"]
display(wl[cols].head(25).rename(columns={
    "obligor_id": "ID", "obligor_name": "Obligor", "sector": "Sector", "rating_then": "Rating (2q ago)",
    "rating_now": "Rating now", "util_prev": "Util. prev q", "util_now": "Util. now", "sector_ratio": "Sector realised / expected",
    "n_flags": "Flags", "reasons": "Rules tripped", "ead_usd_mn": "EAD (USD mn)"}).style.hide(axis="index").format({
    "Util. prev q": "{:.0%}", "Util. now": "{:.0%}", "Sector realised / expected": "{:.1f}x", "EAD (USD mn)": "{:,.1f}"}, na_rep="n/a"))